# Session 1 — Probability Basics

**Goal:** learn to read the number this whole module is building toward — a
probability of disease for one patient — without misreading it. By the end you can
compute a baseline rate, condition it on a measurement, check whether a measurement
carries any information at all, invert a screening test with Bayes' theorem, and
rebuild an overall rate from its parts.

## What this stage does for the system

The eleven sessions after this one assemble a heart-disease risk-screening system:
raw clinical measurements go in, a probability of disease comes out. That output is
useless — worse, dangerous — if it is read wrong, and the single most common way to
read it wrong is to confuse *P(positive test | disease)* with *P(disease | positive
test)*. Those two numbers can differ by a factor of twenty on the same test, purely
because of who is being screened. This session establishes the vocabulary the rest of
the module keeps using: baseline rate, conditional probability, independence, and the
Bayes inversion that turns a test's advertised accuracy into an answer a clinician can
actually act on.

Get this stage wrong and nothing downstream saves you: Session 10's beautifully
cross-validated AUC still produces a number someone will misinterpret at the bedside.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — The baseline rate

Before any measurement is known about a specific patient, the best available estimate
of their disease probability is the rate in the registry as a whole. Every conditional
probability computed later in this notebook is only interesting relative to this
number, so it is worth pinning down first.

In [ ]:
n_patients = len(df)
n_disease = df["target"].sum()
p_disease = df["target"].mean()

print(f"patients:   {n_patients}")
print(f"with disease: {n_disease}")
print(f"P(disease) = {p_disease:.3f}   <- the baseline, before knowing anything about a patient")

**Observe:** `P(disease) = 0.461` — 137 of 297 patients. Close to a coin flip.
**Infer:** this is a *referral* population, not the general public: these are people
who were already sent for angiography because something looked wrong. A general adult
population would run somewhere near 1-2%. That gap is not a data-quality problem, it
is the single most important fact about this registry, and Step 5 shows it changing
the interpretation of an identical test result by an order of magnitude. Any model
trained here inherits this prevalence, and deploying it in a lower-prevalence setting
without adjustment is one of the classic ways a clinically-validated model fails in
the field.

## Step 3 — Conditional probability: does a measurement move the needle?

$P(\text{disease} \mid \text{measurement})$ is the baseline recomputed on the subset of
patients who share that measurement. If it comes out roughly equal to the baseline,
the measurement told us nothing.

In [ ]:
high_chol = df["chol"] > 240  # a common clinical threshold for "high" cholesterol

p_given_high = df.loc[high_chol, "target"].mean()
p_given_normal = df.loc[~high_chol, "target"].mean()

print(f"patients with chol > 240: {high_chol.sum()} of {len(df)}")
print(f"P(disease | chol > 240)  = {p_given_high:.3f}")
print(f"P(disease | chol <= 240) = {p_given_normal:.3f}")
print(f"baseline                 = {df['target'].mean():.3f}")

**Observe:** `0.523` vs `0.397`, against a `0.461` baseline — knowing the cholesterol
group shifts the estimate by roughly ±6 percentage points.
**Infer:** cholesterol carries *some* information, but far less than its reputation
suggests. A 12-point spread between the two groups is a weak signal, and Session 5
will confirm it numerically (`chol` has the second-weakest correlation with `target`
of any column in this registry). Notice also what the threshold did: `chol > 240`
throws away the difference between 245 and 560. That is a modelling decision smuggled
in as a clinical convention, and Session 9 will use `chol` as a continuous input
instead precisely to avoid it.

## Step 4 — Independence: a measurement that carries *no* information

Two events are independent when conditioning on one leaves the other's probability
unchanged — equivalently, when $P(A \cap B) = P(A)\,P(B)$. `fbs` (fasting blood sugar
above 120 mg/dL) makes a good worked example, because it is the kind of variable
everyone assumes must matter.

In [ ]:
p_given_fbs = df.loc[df["fbs"] == 1, "target"].mean()
p_given_no_fbs = df.loc[df["fbs"] == 0, "target"].mean()

joint = ((df["fbs"] == 1) & (df["target"] == 1)).mean()
product = (df["fbs"] == 1).mean() * df["target"].mean()

print(f"P(disease | fbs = 1) = {p_given_fbs:.3f}")
print(f"P(disease | fbs = 0) = {p_given_no_fbs:.3f}")
print()
print(f"P(fbs=1 and disease)   = {joint:.4f}")
print(f"P(fbs=1) * P(disease)  = {product:.4f}")

**Observe:** `0.465` vs `0.461` — a four-thousandths difference — and the joint
probability `0.0673` almost exactly matching the product `0.0668`.
**Infer:** in this registry `fbs` is, for practical purposes, statistically independent
of disease. That is a genuinely useful finding, not a failed one: an input this flat is
a candidate for removal, and carrying it into a model spends a degree of freedom on
noise. Two cautions before you act on it. First, "independent in a sample of 297" is
not "independent in reality" — Session 6 is the machinery for deciding whether a gap
this small is distinguishable from chance at all, and with only 43 `fbs = 1` patients
here it would not be. Second, independence from the target *alone* does not license
dropping a variable in a multivariate model, where a variable can matter only in
combination with another; Session 9 revisits exactly that.

## Step 5 — Bayes' theorem: the direction that actually matters

A screening test is advertised by its **sensitivity** ($P(\text{positive} \mid
\text{disease})$) and **specificity** ($P(\text{negative} \mid \text{no disease})$).
Neither is what a patient holding a positive result wants to know. They want
$P(\text{disease} \mid \text{positive})$ — the *positive predictive value* — and
getting from one to the other requires the baseline rate from Step 2:

$$P(D \mid +) = \frac{P(+ \mid D)\,P(D)}{P(+ \mid D)\,P(D) + P(+ \mid \lnot D)\,P(\lnot D)}$$

In [ ]:
def positive_predictive_value(prevalence, sensitivity, specificity):
    """P(disease | positive test), via Bayes' theorem."""
    p_positive = sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
    return sensitivity * prevalence / p_positive

clinic_prevalence = df["target"].mean()
ppv_clinic = positive_predictive_value(clinic_prevalence, sensitivity=0.90, specificity=0.80)
ppv_population = positive_predictive_value(0.01, sensitivity=0.90, specificity=0.80)

print(f"Same test (90% sensitive, 80% specific), two settings:")
print(f"  this referral registry (prevalence {clinic_prevalence:.3f}): PPV = {ppv_clinic:.3f}")
print(f"  general population     (prevalence 0.010): PPV = {ppv_population:.3f}")
print()
print(f"ratio: {ppv_clinic / ppv_population:.1f}x")

**Observe:** `PPV = 0.794` in the registry versus `0.043` in a general population —
an 18-fold difference from an *identical* test.
**Infer:** this is the number that makes population screening for rare conditions hard:
at 1% prevalence, 96 of every 100 positives are false alarms, and no amount of
tinkering with the model changes that — it is arithmetic, not a modelling deficiency.
The lever that actually moves PPV at low prevalence is specificity, not sensitivity;
re-run the cell with `specificity=0.99` and watch the population PPV jump while the
sensitivity change barely registers. Practically, this is why the system being built
here is scoped as a *referral-clinic* tool: the prevalence baked into the training data
is a precondition for its output being interpretable, and Session 10's held-out
evaluation measures performance only under that same prevalence.

## Step 6 — Law of total probability: rebuilding the whole from its parts

The overall rate is the weighted average of the subgroup rates, weighted by how big
each subgroup is: $P(D) = \sum_i P(D \mid G_i)\,P(G_i)$. This is both a sanity check
on a subgroup breakdown and the mechanism behind the stratified sampling of Session 4.

In [ ]:
df["age_group"] = pd.cut(df["age"], bins=[0, 45, 55, 65, 100],
                         labels=["under 45", "45-55", "55-65", "65+"])

groups = df.groupby("age_group", observed=True).agg(
    n=("target", "size"),
    rate=("target", "mean"),
)
groups["weight"] = groups["n"] / len(df)
print(groups.round(3))

reconstructed = (groups["rate"] * groups["weight"]).sum()
print(f"\nreconstructed P(disease) = {reconstructed:.4f}")
print(f"directly computed   P(disease)  = {df['target'].mean():.4f}")

**Observe:** the reconstruction matches the direct mean to four decimals
(`0.4613`), and the group rates climb `0.246 → 0.376 → 0.627` before dipping to
`0.485` in the `65+` group.
**Infer:** the exact match is arithmetic, not evidence — it holds for any partition, so
treat a mismatch as a bug (usually rows silently dropped by `pd.cut` falling outside
the bins). The interesting part is the non-monotone dip at `65+`. Before reading that
as "risk declines after 65", note the group size: 33 patients, so the rate moves by 3
points for every single patient. Session 4 quantifies exactly how unreliable an
estimate from 33 people is, and Session 6 gives you the test for whether that dip is
distinguishable from noise. It is not — but the point is that you cannot tell from the
table alone, which is why the sessions after this one exist.

## What this session hands to the next one

- **A baseline rate (0.461)** that every later probability is measured against, and a
  standing warning that it is a referral-population rate, not a general one.
- **Conditional probability** as the mechanism behind every "does this input matter"
  question in Sessions 5-8.
- **An independence check** — the informal version of the hypothesis tests in
  Sessions 6-8, which replace "these two numbers look close" with a p-value.
- **Bayes' inversion**, which is what makes Session 10's classifier output
  interpretable rather than just well-scored.

Session 2 takes the next step down: instead of asking about probabilities of events,
it characterises each individual input — `age`, `chol`, `thalach` — as a random
variable with its own mean, spread, and shape.

## Try it yourself

1. Replace the `chol > 240` threshold in Step 3 with `> 200` and `> 300`. At which
   threshold does the conditional probability separate the groups most? What does the
   shrinking subgroup size do to your confidence in that answer?
2. Run the Step 4 independence check on `exang` instead of `fbs`. How does a variable
   that *does* carry information look, side by side with one that does not?
3. In Step 5, find the specificity at which a general-population PPV reaches 0.50.
   Is a test that good realistic?
4. Redo Step 6 partitioning on `cp` (chest-pain type) instead of age groups. Which
   partition produces subgroup rates furthest from the baseline — and is that the same
   as being the most useful input?